# BERT on WikiText — encoder activations, cached to HDF5

Same run as the in-memory example, but with `cache_outputs=True` so
activations stream to an HDF5 file instead of staying in RAM. Downloads on
first run: WikiText-2 (~5 MB) and BERT weights (~440 MB).

In [ ]:
import sys
from pathlib import Path

# Make the repo-local examples._utils package importable when this notebook
# is opened directly from examples/text/, without installing anything extra.
sys.path.insert(0, str(Path.cwd().parents[1]))

from transformers import AutoModelForTokenClassification, AutoTokenizer

from examples._utils.data import activation_loader
from examples._utils.text import WikiTextSamples
from nnact import ActivationPipeline
from nnact._model._hooked import HookedModel

MODEL = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
# A token-classification head gives per-token logits; its weights are
# randomly initialized on top of pretrained BERT, so predictions are
# meaningless here — only the encoder activations are of interest.
model = AutoModelForTokenClassification.from_pretrained(MODEL)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

In [ ]:
dataset = WikiTextSamples(tokenizer, n=256, max_length=64)
print(f"{len(dataset)} passages | input_ids {tuple(dataset[0]['input_ids'].shape)}")
print(f"real tokens in first: {int(dataset[0]['attention_mask'].sum())}")
print(dataset.texts[0][:90], "...")


In [ ]:
hooked = HookedModel(model)

# depth=3 reaches the individual blocks; depth=1 would only show
# "embeddings" and "encoder".
hooked.summary(depth=3).head(12)

In [ ]:
# Embeddings, an early block, a middle block, and the last one.
LAYERS = [
    # "bert.embeddings",
    # "bert.encoder.layer.0",
    "bert.encoder.layer.5",
    # "bert.encoder.layer.11",
]

# cache_outputs=True streams every batch's activations to an HDF5 file at
# run_dir/activations.h5 instead of accumulating them in memory. Passing a
# tokenizer for a "token" pipeline also attaches token-level metadata:
# token_ids and decoded tokens. run_dir also collects run.log and
# run_metadata.json for every run, cached or not.
pipeline = ActivationPipeline(
    model,
    LAYERS,
    output_type="token",
    tokenizer=tokenizer,
    cache_outputs=True,
    run_dir=Path.cwd() / "runs" / "03_bert_wikitext_cache",
)
loader = activation_loader(dataset, batch_size=32)
activations = pipeline.run(loader)
activations.summary()
activations.print_cache_info()

In [ ]:
# Sanity check: the accumulated per-sample token ids must equal each
# passage's real (non-padding) input_ids, in order, and the decoded tokens
# must match what the tokenizer itself produces for those same ids.
offsets = activations.offsets
token_ids, tokens = activations.token_ids, activations.tokens

for i in range(len(dataset)):
    mask = dataset[i]["attention_mask"].bool()
    expected_ids = dataset[i]["input_ids"][mask]

    start, end = int(offsets[i]), int(offsets[i + 1])
    sample_ids = token_ids[start:end]
    sample_tokens = tokens[start:end]

    assert (sample_ids == expected_ids.numpy()).all(), (
        f"sample {i}: token_ids do not match the original dataset"
    )
    assert sample_tokens.tolist() == tokenizer.convert_ids_to_tokens(
        expected_ids.tolist()
    ), f"sample {i}: decoded tokens do not match the tokenizer"

    # Recombining the accumulated tokens should round-trip back to (a
    # normalized form of) the original passage text. BERT's tokenizer
    # lowercases and strips accents/punctuation spacing, and the dataset
    # truncates to max_length, so compare against the tokenizer's own
    # decoding of the truncated ids rather than the raw source text. Special
    # tokens ([CLS]/[SEP]) are skipped on both sides, since decode() drops
    # them but convert_ids_to_tokens() does not.
    real_tokens = [
        token
        for token, token_id in zip(sample_tokens.tolist(), sample_ids.tolist())
        if token_id not in tokenizer.all_special_ids
    ]
    remapped = tokenizer.convert_tokens_to_string(real_tokens)
    expected_text = tokenizer.decode(expected_ids, skip_special_tokens=True)
    assert remapped.strip() == expected_text.strip(), (
        f"sample {i}: recombined tokens do not remap to the original text\n"
        f"  remapped: {remapped!r}\n"
        f"  expected: {expected_text!r}"
    )

print(f"token-level metadata verified for all {len(dataset)} samples")